## The function library — `pyspark.sql.functions`

Most of the data work in this notebook calls into `pyspark.sql.functions` — Spark's library of column-level functions for strings, dates, arrays, conditionals, and aggregations.

A few rules:

- **Import explicitly.** `from pyspark.sql.functions import col, when, sum as _sum, ...`. Or alias the whole module: `import pyspark.sql.functions as F`. Either is fine; we use the explicit form below.
- **JVM-resident functions beat Python UDFs.** Every function in this module runs inside the Spark JVM and benefits from Tungsten code generation. A Python UDF requires a serialization round-trip per row through a Python worker — orders of magnitude slower.
- **Watch for builtin collisions.** `sum`, `min`, `max`, `round`, and `filter` all clash with Python builtins. Import them with an alias (`from pyspark.sql.functions import sum as _sum`) so a stray `sum(...)` call doesn't quietly hit the Python builtin instead.

Notebook 03 already covered `select`, `filter`, `withColumn`, `withColumnRenamed`, `drop`, `cast`, `sort`, `distinct`, `dropDuplicates`, and null handling. This notebook builds on top of those.

## Setup

Two real bank tables from `data/`: `customers` (CSV) and `card_transactions` (Parquet). If you haven't yet, run `generate_bank_data.ipynb` to populate `data/`.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, TimestampType,
)

builder = (
    SparkSession.builder
    .appName("TransformationsAggregations")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "data"

customers_schema = StructType([
    StructField("customer_id",   StringType(), nullable=False),
    StructField("full_name",     StringType()),
    StructField("email",         StringType()),
    StructField("phone",         StringType()),
    StructField("date_of_birth", DateType()),
    StructField("gender",        StringType()),
    StructField("city",          StringType()),
    StructField("state",         StringType()),
    StructField("country",       StringType()),
    StructField("created_at",    TimestampType()),
])

customers = (
    spark.read
    .schema(customers_schema)
    .option("header", "true")
    .csv(f"{DATA_DIR}/customers")
)

card_transactions = spark.read.parquet(f"{DATA_DIR}/card_transactions")

print("customers         :", customers.count(), "rows")
print("card_transactions :", card_transactions.count(), "rows")
card_transactions.printSchema()

## String functions

Vectorized string operations, all running on the JVM:

| Function | Purpose |
|---|---|
| `upper`, `lower` | Case change |
| `trim`, `ltrim`, `rtrim` | Strip whitespace |
| `length(col)` | String length |
| `substring(col, pos, len)` | Extract by position (1-based) |
| `concat`, `concat_ws(sep, ...)` | Join strings (the latter inserts a separator) |
| `split(col, pattern)` | Split into array on a regex |
| `regexp_replace(col, pattern, repl)` | Regex replace |
| `regexp_extract(col, pattern, group)` | Pull out the Nth capture group |
| `lpad`, `rpad` | Pad to fixed width |
| `instr(col, substring)` | 1-based index of substring; 0 if not found |

In [ ]:
from pyspark.sql.functions import (
    col, upper, length, concat_ws, substring, regexp_replace,
)

(
    card_transactions
    .select(
        col("merchant_name"),
        upper(col("merchant_name")).alias("merchant_upper"),
        length(col("merchant_name")).alias("name_len"),
        regexp_replace(col("merchant_name"), r"\s+", "_").alias("merchant_slug"),
        substring(col("transaction_id"), 1, 4).alias("txn_prefix"),
        concat_ws(" | ", col("merchant_category"), col("status")).alias("category_status"),
    )
    .show(5, truncate=False)
)

## Date and timestamp functions

`DateType` stores days since epoch; `TimestampType` stores microseconds. The functions below work on both.

| Function | Purpose |
|---|---|
| `year`, `quarter`, `month`, `dayofmonth`, `dayofweek`, `weekofyear`, `hour`, `minute` | Extract a part |
| `date_format(col, fmt)` | Format as a string (Java SimpleDateFormat patterns) |
| `to_date(col, fmt)`, `to_timestamp(col, fmt)` | Parse a string |
| `datediff(end, start)`, `months_between(end, start)` | Time between two dates |
| `date_add(col, n)`, `date_sub(col, n)`, `add_months(col, n)` | Arithmetic |
| `current_date()`, `current_timestamp()` | Now |
| `unix_timestamp(col)`, `from_unixtime(col)` | Convert to/from epoch seconds |
| `trunc(col, "month")`, `date_trunc("hour", col)` | Truncate to a unit |

In [ ]:
from pyspark.sql.functions import (
    year, month, dayofmonth, date_format, datediff, current_date, trunc,
)

(
    card_transactions
    .select(
        col("transaction_at"),
        year(col("transaction_at")).alias("yr"),
        month(col("transaction_at")).alias("mo"),
        dayofmonth(col("transaction_at")).alias("day"),
        date_format(col("transaction_at"), "yyyy-MM-dd HH:mm").alias("formatted"),
        datediff(current_date(), col("transaction_at").cast("date")).alias("days_ago"),
        trunc(col("transaction_at").cast("date"), "month").alias("month_start"),
    )
    .show(5, truncate=False)
)

## Array functions

`ArrayType` columns are first-class. The function library has both array-builders and array-flatteners.

| Function | Purpose |
|---|---|
| `array(col1, col2, ...)` | Build an array from columns or literals |
| `array_contains(col, value)` | Boolean: does the array contain value? |
| `size(col)` | Number of elements |
| `array_distinct`, `array_union`, `array_intersect`, `array_except` | Set ops |
| `explode(col)` | One row per element (rows with `null` arrays are dropped entirely) |
| `explode_outer(col)` | Same, but `null` arrays survive as one `null`-valued row |
| `posexplode(col)` | Adds an `(index, value)` pair |
| `inline(struct_array)` | Flatten an array of structs into rows |

In [ ]:
from pyspark.sql.functions import array, array_contains, size, explode, when, lit

# Tag each transaction with a list of risk reasons. Always non-empty: 'CLEAN' if nothing else fires.
risk_categorized = card_transactions.withColumn(
    "risk_reasons",
    when(col("is_flagged") & (col("amount") > 50000),
         array(lit("HIGH_AMOUNT"), lit("FRAUD_FLAG")))
    .when(col("is_flagged"),
         array(lit("FRAUD_FLAG")))
    .when(col("amount") > 50000,
         array(lit("HIGH_AMOUNT")))
    .when(col("status") == "DECLINED",
         array(lit("DECLINED_TXN")))
    .otherwise(array(lit("CLEAN"))),
)

# size — how many reasons per transaction?
(
    risk_categorized
    .select("transaction_id", "risk_reasons", size(col("risk_reasons")).alias("n_reasons"))
    .show(5, truncate=False)
)

# array_contains — find FRAUD_FLAG transactions
fraud_arr = risk_categorized.filter(array_contains(col("risk_reasons"), "FRAUD_FLAG"))
print("FRAUD_FLAG transactions:", fraud_arr.count())

# explode — one row per reason per transaction
exploded = risk_categorized.select("transaction_id", explode(col("risk_reasons")).alias("reason"))
exploded.groupBy("reason").count().orderBy("count", ascending=False).show()

## Conditional columns — `when` / `otherwise`

The DataFrame `CASE WHEN`. Chain `.when()` for each branch and finish with `.otherwise()` for the default. **Always** provide `.otherwise()` — without it, unmatched rows become `null` silently.

In [ ]:
tier_df = card_transactions.withColumn(
    "spend_tier",
    when(col("amount") > 50000, "HIGH")
    .when(col("amount") > 10000, "STANDARD")
    .when(col("amount") >  1000, "LOW")
    .otherwise("MICRO"),
)

tier_df.groupBy("spend_tier").count().orderBy("count", ascending=False).show()

## Joins — types, syntax, and the duplicate-column trap

`df1.join(df2, on, how)`. Six `how` values worth knowing:

| `how` | Returns |
|---|---|
| `inner` (default) | Rows where the join key matches in both |
| `left` | All rows from `df1`, with `null` columns where `df2` has no match |
| `right` | All rows from `df2`, with `null` columns where `df1` has no match |
| `full` (or `outer`) | All rows from both, with `null` where the other side is missing |
| `left_semi` | Rows from `df1` *that have a match in `df2`* — no `df2` columns added |
| `left_anti` | Rows from `df1` *that have no match in `df2`* — no `df2` columns added |
| `cross` | Cartesian product (use with caution) |

Two ways to specify the key:

- **String column name** when both sides have the column with the same name — Spark de-duplicates the join column automatically.
- **Boolean expression** (`df1.x == df2.y`) when names differ — both columns survive in the result, which is the duplicate-column trap. Drop or rename one side after the join.

*Broadcast joins, sort-merge vs hash strategy, and skew handling are notebook 07 (Performance & Tuning).*

In [ ]:
from pyspark.sql.functions import sum as _sum, count, round as _round

# Inner join — total spend by city (customer_id is the shared key)
spend_by_city = (
    customers
    .join(card_transactions, "customer_id", "inner")
    .filter(col("status") == "APPROVED")
    .groupBy("city")
    .agg(
        _round(_sum("amount"), 2).alias("total_spend"),
        count("*").alias("n_txns"),
    )
    .orderBy("total_spend", ascending=False)
)
spend_by_city.show(5)

# Left anti — customers with zero card transactions
inactive = customers.join(card_transactions, "customer_id", "left_anti")
print("inactive customers:", inactive.count())
inactive.select("customer_id", "full_name", "city").show(3)

## `groupBy` + `agg`

`groupBy(...)` partitions rows by one or more keys; `agg(...)` applies one or more aggregate functions to each group. Both are lazy; execution waits for an action.

In [ ]:
from pyspark.sql.functions import (
    avg, countDistinct, max as _max, min as _min,
)

# Single-key group: spend per merchant_category (approved only)
(
    card_transactions
    .filter(col("status") == "APPROVED")
    .groupBy("merchant_category")
    .agg(
        count("*").alias("n_txns"),
        _round(_sum("amount"), 2).alias("total_spend"),
        _round(avg("amount"),  2).alias("avg_amount"),
        _max("amount").alias("max_amount"),
        countDistinct("customer_id").alias("unique_customers"),
    )
    .orderBy("total_spend", ascending=False)
    .show(truncate=False)
)

# Multi-key group: spend per (category, status)
(
    card_transactions
    .groupBy("merchant_category", "status")
    .agg(_round(_sum("amount"), 2).alias("total"))
    .orderBy("merchant_category", "status")
    .show(15)
)

## Aggregate function reference

| Function | Notes |
|---|---|
| `count("col")` | Counts non-null values; `count("*")` counts all rows |
| `sum("col")` | Import as `_sum` to dodge Python's `sum` builtin |
| `avg("col")` / `mean("col")` | Mean; nulls are ignored |
| `min("col")`, `max("col")` | Import as `_min`, `_max` |
| `countDistinct("col")` | Exact distinct count — full shuffle |
| `approx_count_distinct("col", rsd=0.05)` | HyperLogLog — much cheaper on large data |
| `collect_list("col")` | All values as an array (preserves duplicates) |
| `collect_set("col")` | Unique values as an array |
| `stddev("col")`, `variance("col")` | Sample stddev / variance |
| `first("col", ignoreNulls=True)`, `last(...)` | First / last value in the group; meaningful only with `orderBy` upstream |

## Filtering groups — Spark's `HAVING` equivalent

Spark has no `HAVING` keyword. Chain `.filter()` after `.agg()` to filter by an aggregated column.

In [ ]:
# Customers with more than 5 approved transactions
heavy_users = (
    card_transactions
    .filter(col("status") == "APPROVED")
    .groupBy("customer_id")
    .agg(count("*").alias("n_approved"))
    .filter(col("n_approved") > 5)            # this is the HAVING
    .orderBy("n_approved", ascending=False)
)

print("heavy users:", heavy_users.count())
heavy_users.show(10)

## Pivot tables

`pivot` rotates a column's distinct values into separate columns. **Always pass the value list explicitly** — `df.pivot("col", [...])` skips the discovery scan that Spark would otherwise run to find distinct values, and it locks the column order in the output.

In [ ]:
# Monthly transaction count by status — explicit value list = fast, deterministic column order
monthly_status = (
    card_transactions
    .withColumn("mo", month(col("transaction_at")))
    .groupBy("mo")
    .pivot("status", ["APPROVED", "DECLINED", "REVERSED"])
    .agg(count("*"))
    .orderBy("mo")
)
monthly_status.show()

## Window functions — concept and spec

A **window function** computes a value *per row* using a frame of neighboring rows. Unlike `groupBy`, the original rows survive — no collapse.

A window spec has three parts:

1. **`partitionBy(...)`** — groups rows. The frame resets at each partition boundary.
2. **`orderBy(...)`** — defines row order within each partition.
3. **Frame** — `rowsBetween(start, end)` (physical row offsets) or `rangeBetween(start, end)` (value offsets along the order-by column). Defaults differ by function: ranking functions ignore the frame; aggregate functions default to the running frame.

The constants `Window.unboundedPreceding`, `Window.currentRow`, `Window.unboundedFollowing` are the frame boundaries.

![Window Function](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-window-function.svg)

## Ranking — `row_number`, `rank`, `dense_rank`, `ntile`

Behavior on ties:

| Function | On ties (1st, 1st, 3rd) |
|---|---|
| `row_number()` | Unique sequential — ties broken arbitrarily: `1, 2, 3` |
| `rank()` | Tied rows share the rank; next rank skips: `1, 1, 3` |
| `dense_rank()` | Tied rows share the rank; no gap: `1, 1, 2` |
| `ntile(n)` | Assigns rows to one of `n` quantile buckets |

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, ntile

# Top approved transaction per customer using row_number
w_top = Window.partitionBy("customer_id").orderBy(col("amount").desc())

top_per_customer = (
    card_transactions
    .filter(col("status") == "APPROVED")
    .withColumn("rn", row_number().over(w_top))
    .filter(col("rn") == 1)
    .select("customer_id", "transaction_id", "amount")
)
top_per_customer.show(5)

# Quartile bucketing — split customers into 4 spend bands
customer_spend = (
    card_transactions
    .filter(col("status") == "APPROVED")
    .groupBy("customer_id")
    .agg(_sum("amount").alias("total_spend"))
)

w_quartile = Window.orderBy(col("total_spend").desc())
(
    customer_spend
    .withColumn("quartile", ntile(4).over(w_quartile))
    .show(10)
)

## `lag` and `lead`

Within a partition, `lag(col, n, default)` looks back `n` rows; `lead(col, n, default)` looks forward `n` rows. Both return the default (or `null`) at the partition boundary.

Useful for delta calculations — "days since previous transaction," "time-to-next-event," running differences.

In [ ]:
from pyspark.sql.functions import lag, to_date

# Days since the customer's previous approved transaction
w_history = Window.partitionBy("customer_id").orderBy("transaction_at")

with_gap = (
    card_transactions
    .filter(col("status") == "APPROVED")
    .withColumn("prev_txn_at", lag(col("transaction_at"), 1).over(w_history))
    .withColumn(
        "days_since_prev",
        datediff(to_date(col("transaction_at")), to_date(col("prev_txn_at"))),
    )
)

(
    with_gap
    .select("customer_id", "transaction_id", "transaction_at", "prev_txn_at", "days_since_prev")
    .orderBy("customer_id", "transaction_at")
    .show(10, truncate=False)
)

## Running aggregates — `rowsBetween` vs `rangeBetween`

Both define the frame, but differently:

- **`rowsBetween(start, end)`** — counts physical row positions relative to the current row.
- **`rangeBetween(start, end)`** — counts the value range of the order-by column. Useful when the order-by column is numeric or a date and you want "all rows within ±7 days of the current row."

Boundary constants:

- `Window.unboundedPreceding` — from the start of the partition
- `Window.currentRow` — the current row
- `Window.unboundedFollowing` — to the end of the partition

Three frames worth memorizing:

- **Cumulative**: `rowsBetween(Window.unboundedPreceding, Window.currentRow)`
- **N-row trailing window**: `rowsBetween(-N, Window.currentRow)`
- **Centered N-row window**: `rowsBetween(-N, N)`

In [ ]:
# Cumulative spend per customer (running total over time)
w_cum = (
    Window.partitionBy("customer_id")
    .orderBy("transaction_at")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# 3-row trailing average per customer (current row + 2 preceding)
w_avg3 = (
    Window.partitionBy("customer_id")
    .orderBy("transaction_at")
    .rowsBetween(-2, Window.currentRow)
)

(
    card_transactions
    .filter(col("status") == "APPROVED")
    .withColumn("cum_spend",  _round(_sum("amount").over(w_cum),  2))
    .withColumn("avg_last_3", _round(avg("amount").over(w_avg3), 2))
    .select("customer_id", "transaction_at", "amount", "cum_spend", "avg_last_3")
    .orderBy("customer_id", "transaction_at")
    .show(10, truncate=False)
)

In [ ]:
spark.stop()